# 11. Cleaning: dunnhumby (The Complete Journey)

The raw files `transaction_data.csv`, `product.csv` and `hh_demographic.csv` are read from `archive/`, and the cleaned tables are written to `clean/dunnhumby__*.csv`. The Online Retail II notebook produces the same files and columns, so notebook 13 can read both in the same way.

Design choices:
- item = `SUB_COMMODITY_DESC` (2,329 categories)
- customer = `household_key`
- period = 4 weeks from `WEEK_NO`, keep weeks 4-24 -> periods 0-20
- edges are binary
- split: train 0-14, validation 15-17, test 18-20
- demographics: every household gets a feature vector, missing values are Unknown, and a `has_demographics` flag marks the 801 households that have real ones

In [1]:
import numpy as np, pandas as pd, json
from pathlib import Path

SLUG   = "dunnhumby"
SRC    = Path(".") if Path("transaction_data.csv").exists() else Path("archive")
OUT    = Path("clean"); OUT.mkdir(exist_ok=True)

ITEM_COL        = "SUB_COMMODITY_DESC"
PERIOD_WEEKS    = 4
KEEP_PERIODS    = (4, 24)          # drop panel ramp-up (0–3) and the 2-week tail (25)
CYCLE_LEN       = 13               # 13 four-week periods ≈ 1 year
SPLIT           = {"train": range(0, 15), "val": (15, 16, 17), "test": (18, 19, 20)}
NON_PRODUCT_DEPTS = {"KIOSK-GAS","MISC SALES TRAN","COUPON","MISC. TRANS.","POSTAL CENTER",
                     "CHARITABLE CONT","CNTRL/STORE SUP","PHARMACY SUPPLY","VIDEO RENTAL",
                     "PHOTO","GM MERCH EXP","RX"}
SEED = 42; np.random.seed(SEED)
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)

LEDGER = []
def note(step, n, detail=""):
    prev = LEDGER[-1][1] if LEDGER else n
    LEDGER.append((step, n, n - prev, detail))
    print(f"{step:<44} rows={n:>9,} ({n-prev:+,})  {detail}")

txn  = pd.read_csv(SRC/"transaction_data.csv")
prod = pd.read_csv(SRC/"product.csv")
demo = pd.read_csv(SRC/"hh_demographic.csv")
note("00  raw transaction_data", len(txn), f"{txn.household_key.nunique():,} households")

00  raw transaction_data                     rows=2,595,732 (+0)  2,500 households


## 1. Cleaning steps
Each step prints how many rows it removes.

In [2]:
d = txn.merge(prod[["PRODUCT_ID","DEPARTMENT","COMMODITY_DESC","SUB_COMMODITY_DESC"]],
              on="PRODUCT_ID", how="left")
assert d.DEPARTMENT.notna().all(), "a PRODUCT_ID has no entry in product.csv"

# D1: returns and free/zero-value lines are not purchases
d = d[(d.QUANTITY > 0) & (d.SALES_VALUE > 0)]
note("01  dropped returns / zero-value lines", len(d))

# D2: gas kiosk, coupons, postal, charity: not purchasable product categories
d = d[~d.DEPARTMENT.isin(NON_PRODUCT_DEPTS)]
note("02  dropped non-product departments", len(d), f"{d.DEPARTMENT.nunique()} departments remain")

# D3: the QUANTITY blow-ups (>1000 units) were almost all KIOSK-GAS; confirm they are gone
assert (d.QUANTITY > 10_000).sum() == 0, "extreme quantities survived the department filter"
print(f"    max QUANTITY now {d.QUANTITY.max():,} (was {txn.QUANTITY.max():,})")

# D4: 4-week periods; drop the recruitment ramp-up and the 2-week tail
d["period_idx"] = (d.WEEK_NO - 1) // PERIOD_WEEKS
lo, hi = KEEP_PERIODS
d = d[d.period_idx.between(lo, hi)].copy()
d["period_idx"] -= lo
note("04  kept complete periods only", len(d),
     f"periods 0–{d.period_idx.max()} ({d.household_key.nunique():,} households)")

# String hygiene: defensive; the labels become graph nodes
for c in ["DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]:
    d[c] = d[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    assert d[c].str.lower().nunique() == d[c].nunique(), f"{c} has case-variant duplicates"
print(f"    item vocabulary: {d[ITEM_COL].nunique():,} {ITEM_COL}")

# BASKET_ID integrity (it is a valid key here, unlike dataset 1's Transaction_ID)
g = d.groupby("BASKET_ID")
assert g.household_key.nunique().max() == 1 and g.DAY.nunique().max() == 1
print(f"    {d.BASKET_ID.nunique():,} baskets, each one household and one day")

01  dropped returns / zero-value lines       rows=2,576,815 (-18,917)  


02  dropped non-product departments          rows=2,546,354 (-30,461)  32 departments remain
    max QUANTITY now 144 (was 89,638)


04  kept complete periods only               rows=2,300,921 (-245,433)  periods 0–20 (2,492 households)


    item vocabulary: 2,329 SUB_COMMODITY_DESC
    225,922 baskets, each one household and one day


## 2. Calendar
`DAY` is relative (1-711) but the same for all households, so periods line up. 13 periods is about a year.

In [3]:
split_of = {p: name for name, rng in SPLIT.items() for p in rng}
periods  = sorted(d.period_idx.unique())
cycle    = np.array(periods) % CYCLE_LEN
calendar = pd.DataFrame({
    "period_idx":  periods,
    "label":       [f"P{p:02d} (days {lo*PERIOD_WEEKS*7 + p*28 + 1}–{lo*PERIOD_WEEKS*7 + (p+1)*28})" for p in periods],
    "cycle_pos":   cycle,
    "cycle_sin":   np.sin(2*np.pi*cycle/CYCLE_LEN).round(6),
    "cycle_cos":   np.cos(2*np.pi*cycle/CYCLE_LEN).round(6),
    "elapsed":     periods,
    "split":       [split_of[p] for p in periods],
})
print(calendar.to_string(index=False))

 period_idx              label  cycle_pos  cycle_sin  cycle_cos  elapsed split
          0 P00 (days 113–140)          0   0.000000   1.000000        0 train
          1 P01 (days 141–168)          1   0.464723   0.885456        1 train
          2 P02 (days 169–196)          2   0.822984   0.568065        2 train
          3 P03 (days 197–224)          3   0.992709   0.120537        3 train
          4 P04 (days 225–252)          4   0.935016  -0.354605        4 train
          5 P05 (days 253–280)          5   0.663123  -0.748511        5 train
          6 P06 (days 281–308)          6   0.239316  -0.970942        6 train
          7 P07 (days 309–336)          7  -0.239316  -0.970942        7 train
          8 P08 (days 337–364)          8  -0.663123  -0.748511        8 train
          9 P09 (days 365–392)          9  -0.935016  -0.354605        9 train
         10 P10 (days 393–420)         10  -0.992709   0.120537       10 train
         11 P11 (days 421–448)         11  -0.822984

## 3. Integer indices
Customers and items mapped to 0..n-1. Sorted first, so the mapping is the same every run.

In [4]:
def remap(values, idx_col, label_col):
    """Sorted unique values -> 0..n-1. Sorting makes the mapping deterministic across runs."""
    uniq = sorted(pd.Series(values).dropna().unique())
    return {v: i for i, v in enumerate(uniq)}, pd.DataFrame({idx_col: range(len(uniq)), label_col: uniq})

cust_map, customer_id_map = remap(d.household_key, "customer_idx", "customer_id")
item_map,  item_id_map    = remap(d[ITEM_COL],     "item_idx",     "item_label")
d["customer_idx"] = d.household_key.map(cust_map).astype("int32")
d["item_idx"]     = d[ITEM_COL].map(item_map).astype("int32")

N_CUST, N_ITEM = len(cust_map), len(item_map)
d["item_node_id"] = d.item_idx + N_CUST          # unified node space for single-embedding models

# carry the parent levels onto the item map: lets results be grouped by department
parents = (d.drop_duplicates(ITEM_COL).set_index(ITEM_COL)[["DEPARTMENT","COMMODITY_DESC"]]
             .rename(columns={"DEPARTMENT":"parent_department","COMMODITY_DESC":"parent_commodity"}))
item_id_map = item_id_map.join(parents, on="item_label")
item_id_map["description"] = item_id_map.item_label          # already human-readable here
item_id_map = item_id_map[["item_idx","item_label","description","parent_department","parent_commodity"]]

print(f"customers {N_CUST:,} -> idx 0..{N_CUST-1}")
print(f"items     {N_ITEM:,} -> idx 0..{N_ITEM-1}  (node ids {N_CUST}..{N_CUST+N_ITEM-1})")
assert (d.customer_idx.map(dict(zip(customer_id_map.customer_idx, customer_id_map.customer_id)))
        == d.household_key).all(), "index mapping does not round-trip"
item_id_map.head(3)

customers 2,492 -> idx 0..2491
items     2,329 -> idx 0..2328  (node ids 2492..4820)


,item_idx,item_label,description,parent_department,parent_commodity
0,0,*ATH ACCES:TOWEL BARS/SOAP D,*ATH ACCES:TOWEL BARS/SOAP D,DRUG GM,HARDWARE SUPPLIES
1,1,*ATTERIES:CAMERA/FLASH/WATCH,*ATTERIES:CAMERA/FLASH/WATCH,DRUG GM,BATTERIES
2,2,*BOYS/GIRLS MISC TOYS,*BOYS/GIRLS MISC TOYS,DRUG GM,TOYS AND GAMES


## 4. Edge list
One row per (period, customer, item). `weight` is binary; the other columns (counts, quantity, revenue) are kept for later use.

In [5]:
d["revenue"] = d.SALES_VALUE.round(2)
interactions = (d.groupby(["period_idx","customer_idx","item_idx"], observed=True)
                  .agg(n_lineitems=("QUANTITY","size"), n_baskets=("BASKET_ID","nunique"),
                       total_quantity=("QUANTITY","sum"), total_revenue=("revenue","sum"))
                  .reset_index())
interactions["total_revenue"] = interactions.total_revenue.round(2)
interactions["log_quantity"]  = np.log1p(interactions.total_quantity).round(4)
interactions["weight"]        = 1

print(f"{len(d):,} line items -> {len(interactions):,} edges "
      f"({len(d)/len(interactions):.1f} line items per edge)")
print(f"{len(interactions)//d.period_idx.nunique():,} edges per graph over {d.period_idx.nunique()} periods")
interactions.head(3)

2,300,921 line items -> 1,541,479 edges (1.5 line items per edge)
73,403 edges per graph over 21 periods


,period_idx,customer_idx,item_idx,n_lineitems,n_baskets,total_quantity,total_revenue,log_quantity,weight
0,0,0,32,1,1,1,2.99,0.6931,1
1,0,0,45,3,3,3,10.45,1.3863,1
2,0,0,153,2,1,2,2.58,1.0986,1


## 5. Customer features
Only 801 households have demographics. Everyone gets a vector; missing values are Unknown and flagged. Only static attributes are used, because lifetime totals would leak information from future periods.

In [6]:
DEMO_COLS = ["AGE_DESC","MARITAL_STATUS_CODE","INCOME_DESC","HOMEOWNER_DESC",
             "HH_COMP_DESC","HOUSEHOLD_SIZE_DESC","KID_CATEGORY_DESC"]

cf = customer_id_map.copy()
cf = cf.merge(demo.rename(columns={"household_key":"customer_id"}), on="customer_id", how="left")
cf["has_demographics"] = cf[DEMO_COLS[0]].notna().astype("int8")
cf[DEMO_COLS] = cf[DEMO_COLS].fillna("Unknown")

onehot = pd.get_dummies(cf[DEMO_COLS].astype(str), prefix=DEMO_COLS, dtype="int8")
FEATURE_COLS = list(onehot.columns)
cf = pd.concat([cf, onehot], axis=1)

# split-construction metadata: NOT features
activity = (interactions.groupby("customer_idx")
              .agg(first_period_idx=("period_idx","min"), last_period_idx=("period_idx","max"),
                   n_active_periods=("period_idx","nunique")).reset_index())
customer_features = cf.merge(activity, on="customer_idx").sort_values("customer_idx")

print(f"{len(customer_features):,} customers | {customer_features.has_demographics.sum():,} with real "
      f"demographics ({customer_features.has_demographics.mean()*100:.0f}%)")
print(f"feature matrix: {len(FEATURE_COLS)} one-hot dimensions")
print(f"NOT features: customer_id, has_demographics, first_period_idx, last_period_idx, n_active_periods")

2,492 customers | 801 with real demographics (32%)
feature matrix: 46 one-hot dimensions
NOT features: customer_id, has_demographics, first_period_idx, last_period_idx, n_active_periods


## 6. Leakage checks

In [7]:
# G1: every edge's activity lies inside its own period
chk = d.groupby(["period_idx","customer_idx","item_idx"], observed=True).WEEK_NO.agg(["min","max"])
assert (((chk["min"]-1)//PERIOD_WEEKS - KEEP_PERIODS[0]) == chk.index.get_level_values(0)).all()
assert (((chk["max"]-1)//PERIOD_WEEKS - KEEP_PERIODS[0]) == chk.index.get_level_values(0)).all()
print("G1 PASS  every edge's activity lies within its labelled period")

# G2: splits chronological, disjoint, complete
tr, va, te = [sorted(SPLIT[k]) for k in ("train","val","test")]
assert max(tr) < min(va) and max(va) < min(te)
assert set(tr) | set(va) | set(te) == set(periods)
print("G2 PASS  splits chronological, disjoint, covering all", len(periods), "periods")

# G3: no model-facing feature varies over time
tv = [c for c in FEATURE_COLS
      if d[["customer_idx"]].merge(customer_features[["customer_idx", c]], on="customer_idx")
          .groupby("customer_idx")[c].nunique().max() > 1]
assert not tv, f"time-varying feature columns: {tv}"
print("G3 PASS  every model-facing feature is static")

# G4: the reference historical-feature function reads only periods <= t
def historical_affinity(edges, t, window=3):
    """Leakage-safe affinity features as of period t. Reads only period_idx <= t."""
    past = edges[edges.period_idx <= t]
    if past.empty:
        return pd.DataFrame(columns=["customer_idx","item_idx","lifetime_count","recency",
                                     "recent_count","item_share"])
    f = (past.groupby(["customer_idx","item_idx"], observed=True)
             .agg(lifetime_count=("period_idx","size"), last_seen=("period_idx","max")).reset_index())
    f["recency"] = t - f.last_seen
    rc = (past[past.period_idx > t - window].groupby(["customer_idx","item_idx"], observed=True)
              .size().reset_index(name="recent_count"))
    f = f.merge(rc, on=["customer_idx","item_idx"], how="left")
    f["recent_count"] = f.recent_count.fillna(0).astype(int)
    f["item_share"] = (f.lifetime_count /
                       f.groupby("customer_idx").lifetime_count.transform("sum")).round(4)
    return f.drop(columns="last_seen")

t_demo = 10
a = historical_affinity(interactions, t_demo)
b = historical_affinity(interactions[interactions.period_idx <= t_demo], t_demo)
key = ["customer_idx","item_idx"]
assert a.sort_values(key).reset_index(drop=True).equals(b.sort_values(key).reset_index(drop=True))
print(f"G4 PASS  historical_affinity(t={t_demo}) unchanged when future periods are deleted "
      f"({len(a):,} rows)")

G1 PASS  every edge's activity lies within its labelled period
G2 PASS  splits chronological, disjoint, covering all 21 periods


G3 PASS  every model-facing feature is static


G4 PASS  historical_affinity(t=10) unchanged when future periods are deleted (386,162 rows)


## 7. Export

In [8]:
TXN_COLS = ["BASKET_ID","customer_idx","household_key","period_idx","DAY","WEEK_NO",
            "PRODUCT_ID","item_idx","item_node_id","SUB_COMMODITY_DESC","COMMODITY_DESC",
            "DEPARTMENT","QUANTITY","revenue","STORE_ID"]
transactions_clean = d[TXN_COLS].rename(columns={"household_key":"customer_id",
    "SUB_COMMODITY_DESC":"item_label","COMMODITY_DESC":"parent_commodity",
    "DEPARTMENT":"parent_department","QUANTITY":"quantity","revenue":"revenue",
    "STORE_ID":"store_id","PRODUCT_ID":"product_id","BASKET_ID":"basket_id"}) \
    .sort_values(["period_idx","customer_idx","basket_id"]).reset_index(drop=True)

ARTIFACTS = {"transactions_clean": transactions_clean,
             "interactions_monthly": interactions,
             "customer_features": customer_features,
             "customer_id_map": customer_id_map,
             "item_id_map": item_id_map,
             "calendar": calendar}
for name, frame in ARTIFACTS.items():
    frame.to_csv(OUT/f"{SLUG}__{name}.csv", index=False)

META = {"dataset":"dunnhumby The Complete Journey", "slug":SLUG,
        "item_level":ITEM_COL, "item_meaning":"product sub-commodity category",
        "n_customers":int(N_CUST), "n_items":int(N_ITEM), "n_periods":int(len(periods)),
        "period_length":"4 weeks", "calendar_type":"relative (shared DAY index 1-711)",
        "cycle_len":CYCLE_LEN, "item_node_offset":int(N_CUST),
        "n_edges":int(len(interactions)), "default_edge_weight":"binary",
        "split":{k:list(v) for k,v in SPLIT.items()},
        "feature_columns":FEATURE_COLS,
        "not_features":["customer_id","has_demographics","first_period_idx","last_period_idx",
                        "n_active_periods"],
        "feature_coverage":float(customer_features.has_demographics.mean()),
        "feature_note":"7 demographic fields, present for 801 of 2,492 households; "
                       "the rest encoded Unknown and flagged by has_demographics"}
json.dump(META, open(OUT/f"{SLUG}__metadata.json","w"), indent=2)

print(pd.DataFrame([{"file":f"{SLUG}__{n}.csv","rows":len(f),"cols":f.shape[1],
                     "MB":round((OUT/f"{SLUG}__{n}.csv").stat().st_size/1024**2,1)}
                    for n,f in ARTIFACTS.items()]).to_string(index=False))
print(f"\n+ {SLUG}__metadata.json   ->  {OUT.resolve()}")

                               file    rows  cols    MB
  dunnhumby__transactions_clean.csv 2300921    15 223.4
dunnhumby__interactions_monthly.csv 1541479     9  46.3
   dunnhumby__customer_features.csv    2492    59   0.4
     dunnhumby__customer_id_map.csv    2492     2   0.0
         dunnhumby__item_id_map.csv    2329     5   0.1
            dunnhumby__calendar.csv      21     7   0.0

+ dunnhumby__metadata.json   ->  clean


## 8. Re-read from disk and check

In [9]:
R = {n: pd.read_csv(OUT/f"{SLUG}__{n}.csv") for n in ARTIFACTS}
I, F, T, C = R["interactions_monthly"], R["customer_features"], R["transactions_clean"], R["calendar"]

act = {p:set(g.customer_idx) for p,g in I.groupby("period_idx")}; ps = sorted(act)
sup = sum(len(act[a]&act[b]) for a,b in zip(ps,ps[1:]))
prs = {p:set(zip(g.customer_idx,g.item_idx)) for p,g in I.groupby("period_idx")}
hit = tot = 0
for a,b in zip(ps,ps[1:]):
    both = act[a]&act[b]; x = {q for q in prs[a] if q[0] in both}
    hit += len(x & {q for q in prs[b] if q[0] in both}); tot += len(x)

checks = {
 "no nulls in any artifact":           all(not f.isna().any().any() for f in [I,F,C]),
 "customer_idx contiguous from 0":     sorted(F.customer_idx) == list(range(len(F))),
 "edge indices resolve to real nodes": I.customer_idx.max() < len(F) and I.item_idx.max() < len(R["item_id_map"]),
 "weight is binary everywhere":        (I.weight == 1).all(),
 "periods 0..20 present":              sorted(I.period_idx.unique()) == list(range(21)),
 "every period has a calendar row":    set(I.period_idx) == set(C.period_idx),
 "id maps round-trip":                 T.merge(R["customer_id_map"], on="customer_idx").eval("customer_id_x == customer_id_y").all(),
 "items 2,329":                        len(R["item_id_map"]) == 2329,
 "customers 2,492":                    len(F) == 2492,
 "edges 1,541,479":                    len(I) == 1541479,
 "supervision 35,489":                 sup == 35489,
 "repeat rate 0.318":                  round(hit/tot, 3) == 0.318,
 "test window 1,797 customers":        len(act[19] & act[20]) == 1797,
 "demographics flag on 801":           F.has_demographics.sum() == 801,
}
for k, v in checks.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())
print(f"\n{len(txn):,} raw rows -> {len(T):,} line items -> {len(I):,} edges over {len(ps)} periods")

  PASS  no nulls in any artifact
  PASS  customer_idx contiguous from 0
  PASS  edge indices resolve to real nodes
  PASS  weight is binary everywhere
  PASS  periods 0..20 present
  PASS  every period has a calendar row
  PASS  id maps round-trip
  PASS  items 2,329
  PASS  customers 2,492
  PASS  edges 1,541,479
  PASS  supervision 35,489
  PASS  repeat rate 0.318
  PASS  test window 1,797 customers
  PASS  demographics flag on 801

2,595,732 raw rows -> 2,300,921 line items -> 1,541,479 edges over 21 periods


Output: `clean/dunnhumby__*.csv` and the metadata json. `transactions_clean.csv` (about 230 MB) is only needed for the commodity-level version in notebook 13.